# 2. Calidad, integridad y sesgos de medición

El análisis numérico utiliza el periodo completo de desarrollo **2000–2021**. Los ajustes de escaladores y modelos se realizan dentro de cada fold.

La evaluación de calidad comprende continuidad temporal, unicidad de las claves, valores faltantes, finitud y rangos físicos. Las incidencias se conservan con indicadores de calidad para su interpretación y contraste con las definiciones de las fuentes.

El precio de XM constituye una medida administrativa sujeta a revisión. En las variables meteorológicas, la resolución de rejilla y el carácter retrospectivo del reanálisis introducen diferencias respecto de mediciones instrumentales locales. La cuantificación de ese error requiere observaciones de estaciones emparejadas, ausentes en el conjunto analizado.


In [1]:
%matplotlib inline
from pathlib import Path

ROOT = next(
    directory for directory in [Path.cwd(), *Path.cwd().parents]
    if (directory / "data/manifest.json").exists()
)

from IPython.display import Markdown, display
import json
import numpy as np
import pandas as pd

In [2]:
EDA_SPLITS = {'train': (2000, 2017), 'validation': (2018, 2021), 'test': (2022, 2025)}
SPLITS = EDA_SPLITS
TIME = 'timestamp_start_colombia'
RESULTS = ROOT / 'results'
PIPELINE_DEFAULTS = {'method': 'split_train_val_forwardChaining', 'n_splits': 5, 'development_start': '2000-01-01T00:00:00-05:00', 'validation_start': '2018-01-01T00:00:00-05:00', 'development_end_exclusive': '2022-01-01T00:00:00-05:00', 'test_start': '2022-01-01T00:00:00-05:00', 'test_end_exclusive': '2026-01-01T00:00:00-05:00', 'lookback_hours': 168, 'horizon_hours': 24, 'stride_hours': 24, 'hourly_release_lag_hours': 24, 'hydrology_release_lag_days': 2, 'include_calendar': True, 'seed': 11037, 'availability_scenario': 'retrospective_assumed_release_lags'}
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'figures').mkdir(exist_ok=True)
(RESULTS / 'tables').mkdir(exist_ok=True)
pd.set_option('display.max_rows', 35)
pd.set_option('display.max_columns', 14)
pd.set_option('display.float_format', lambda x: f'{x:.5g}')

In [3]:
def decision(preprocess, architecture, validation, risk):
    display(Markdown('### Implicaciones del análisis\n\n' + '\n'.join(('- **' + a + ':** ' + b for a, b in [('Preprocesamiento', preprocess), ('Arquitectura', architecture), ('Validación', validation), ('Riesgos', risk)]))))

In [4]:
def table(frame, name):
    frame.to_csv(RESULTS / 'tables' / (name + '.csv'), index=True)
    display(frame)
    return None

In [5]:
def load_numeric(split='train', table='hourly', *, allow_test=False):
    if split not in SPLITS:
        raise ValueError('Unknown split')
    if split == 'test' and (not allow_test):
        raise PermissionError('El EDA no puede consultar valores de prueba.')
    a, b = SPLITS[split]
    folder = 'hourly' if table == 'hourly' else 'hydrology'
    frame = pd.concat([pd.read_csv(ROOT / f'data/numeric/{folder}/{year}.csv.gz') for year in range(a, b + 1)], ignore_index=True)
    if table == 'hourly':
        frame.index = pd.DatetimeIndex(pd.to_datetime(frame[TIME], utc=True)).tz_convert('America/Bogota')
    else:
        frame.index = pd.DatetimeIndex(pd.to_datetime(frame['date'])).tz_localize('America/Bogota')
    frame.index.name = 'tiempo_colombia'
    return frame

In [6]:
def load_development_data(table='hourly'):
    return pd.concat([load_numeric('train', table=table), load_numeric('validation', table=table)]).sort_index()

In [7]:
def pipeline_config(overrides=None):
    config = dict(PIPELINE_DEFAULTS)
    if overrides:
        unknown = set(overrides) - set(config)
        if unknown:
            raise ValueError('Opciones desconocidas: ' + str(sorted(unknown)))
        config.update(overrides)
    for field in ['n_splits', 'lookback_hours', 'horizon_hours', 'stride_hours']:
        if not isinstance(config[field], int) or isinstance(config[field], bool) or config[field] < 1:
            raise ValueError('Se requiere un entero positivo: ' + field)
    for field in ['hourly_release_lag_hours', 'hydrology_release_lag_days']:
        if not isinstance(config[field], int) or isinstance(config[field], bool) or config[field] < 0:
            raise ValueError('Rezago inválido: ' + field)
    if config['hydrology_release_lag_days'] < 1:
        raise ValueError('La hidrología diaria requiere al menos el cierre del día.')
    for field in ['development_start', 'validation_start', 'development_end_exclusive', 'test_start', 'test_end_exclusive']:
        if pd.Timestamp(config[field]).tzinfo is None:
            raise ValueError('Se requiere zona horaria: ' + field)
    if config['method'] != 'split_train_val_forwardChaining':
        raise ValueError('Método no autorizado')
    if pd.Timestamp(config['test_start']) != pd.Timestamp(PIPELINE_DEFAULTS['test_start']) or config['test_end_exclusive'] != PIPELINE_DEFAULTS['test_end_exclusive']:
        raise ValueError('El periodo de prueba está congelado.')
    if not pd.Timestamp(config['development_start']) < pd.Timestamp(config['validation_start']) < pd.Timestamp(config['development_end_exclusive']) <= pd.Timestamp(config['test_start']):
        raise ValueError('Cortes temporales inválidos')
    if config['stride_hours'] % 24:
        raise ValueError('Los orígenes deben conservar el paso diario.')
    return config

In [8]:
def load_pipeline_config(path=None):
    path = ROOT / 'configs/training_pipeline.json' if path is None else path
    return pipeline_config(json.loads(path.read_text(encoding='utf-8')))

In [9]:
def load_eda_development(table='hourly'):
    """Desarrollo completo; no cambia las particiones que usa el entrenamiento."""
    config = load_pipeline_config()
    frame = load_development_data(table)
    if len(frame) and frame.index.max() >= pd.Timestamp(config['test_start']):
        raise PermissionError('La fuente contiene fechas de prueba.')
    start = pd.Timestamp(config['development_start'])
    end = pd.Timestamp(config['development_end_exclusive'])
    frame = frame.loc[(frame.index >= start) & (frame.index < end)]
    if frame.empty or frame.index.max() >= pd.Timestamp(config['test_start']):
        raise PermissionError('El EDA sólo admite desarrollo anterior a prueba.')
    return frame

In [10]:
frame = load_eda_development()
hydro = load_eda_development(table='hydrology')
numeric = frame.select_dtypes('number').drop(columns=['xm_hour'], errors='ignore')
expected = pd.date_range('2000-01-01', '2022-01-01', freq='h', inclusive='left', tz='America/Bogota')
quality = pd.DataFrame({'comprobación': ['horas esperadas', 'horas recibidas', 'horas ausentes', 'claves horarias duplicadas', 'valores numéricos faltantes', 'valores no finitos', 'filas hidrológicas', 'claves diarias duplicadas'], 'resultado': [len(expected), len(frame), len(expected.difference(frame.index)), frame.index.duplicated().sum(), numeric.isna().sum().sum(), (~np.isfinite(numeric.to_numpy())).sum(), len(hydro), hydro.index.duplicated().sum()]})
table(quality.set_index('comprobación'), 'calidad_desarrollo')

,resultado
comprobación,
horas esperadas,192864
horas recibidas,192864
horas ausentes,0
claves horarias duplicadas,0
valores numéricos faltantes,0
valores no finitos,0
filas hidrológicas,8036
claves diarias duplicadas,0


In [11]:
stats_table = numeric.agg(['count', 'min', 'max', 'median']).T
stats_table['nulos'] = numeric.isna().sum()
table(stats_table, 'rangos_desarrollo')

,count,min,max,median,nulos
precio_bolsa_cop_kwh,1.9286e+05,21.531,2821.5,93.246,0
demanda_real_kwh,1.9286e+05,4.1315e+05,1.0886e+07,6.489e+06,0
demanda_comercial_kwh,1.9286e+05,1.2123e+05,1.1026e+07,6.598e+06,0
temperature_2m_c_bogota,1.9286e+05,3.2,26.4,14.1,0
relative_humidity_2m_pct_bogota,1.9286e+05,6,100,82,0
precipitation_preceding_hour_mm_bogota,1.9286e+05,0,14.5,0,0
temperature_2m_c_medellin,1.9286e+05,12.7,32.5,20.1,0
relative_humidity_2m_pct_medellin,1.9286e+05,23,100,91,0
precipitation_preceding_hour_mm_medellin,1.9286e+05,0,17,0,0
temperature_2m_c_cali,1.9286e+05,14.5,31.2,21.8,0


In [12]:
physical = {}
for col in numeric:
    x = numeric[col]
    if 'humidity' in col:
        physical[col] = int(((x < 0) | (x > 100)).sum())
    elif 'temperature' in col:
        physical[col] = int(((x < -90) | (x > 60)).sum())
    elif 'precipitation' in col:
        physical[col] = int((x < 0).sum())
    elif 'demanda' in col:
        physical[col] = int((x < 0).sum())
table(pd.Series(physical, name='fuera_de_rango').to_frame(), 'controles_fisicos_desarrollo')

,fuera_de_rango
demanda_real_kwh,0
demanda_comercial_kwh,0
temperature_2m_c_bogota,0
relative_humidity_2m_pct_bogota,0
precipitation_preceding_hour_mm_bogota,0
temperature_2m_c_medellin,0
relative_humidity_2m_pct_medellin,0
precipitation_preceding_hour_mm_medellin,0
temperature_2m_c_cali,0
relative_humidity_2m_pct_cali,0


In [13]:
flags = hydro[(hydro['llenado_valor_original'] < 0) | (hydro['llenado_valor_original'] > 1)]
table(flags, 'llenado_fuera_intervalo_desarrollo')

,date,llenado_valor_original,reserva_util_kwh,capacidad_util_kwh,aportes_kwh_dia
tiempo_colombia,,,,,
2000-01-01 00:00:00-05:00,2000-01-01,1.4914,1.0808e+10,7.2467e+09,5.6677e+07
2000-01-02 00:00:00-05:00,2000-01-02,1.4887,1.0788e+10,7.2467e+09,5.4645e+07
2000-01-03 00:00:00-05:00,2000-01-03,1.4839,1.0753e+10,7.2467e+09,5.7488e+07
2000-01-04 00:00:00-05:00,2000-01-04,1.4792,1.072e+10,7.2467e+09,5.8418e+07
2000-01-05 00:00:00-05:00,2000-01-05,1.4751,1.069e+10,7.2467e+09,6.304e+07
2000-01-06 00:00:00-05:00,2000-01-06,1.4713,1.0662e+10,7.2467e+09,5.9743e+07
2000-01-07 00:00:00-05:00,2000-01-07,1.4682,1.0639e+10,7.2467e+09,6.4109e+07
2000-01-08 00:00:00-05:00,2000-01-08,1.4684,1.0641e+10,7.2467e+09,7.4865e+07
2000-01-09 00:00:00-05:00,2000-01-09,1.4685,1.0642e+10,7.2467e+09,8.0338e+07


In [14]:
display(Markdown(f'Se conservan **{len(flags)}** fechas de llenado fuera de [0,1] en desarrollo. Son banderas de definición/rango, no una corrección automática. La ausencia de nulos en la tabla numérica no acredita ausencia de error instrumental, sesgo de rejilla o revisiones.'))

Se conservan **31** fechas de llenado fuera de [0,1] en desarrollo. Son banderas de definición/rango, no una corrección automática. La ausencia de nulos en la tabla numérica no acredita ausencia de error instrumental, sesgo de rejilla o revisiones.

In [15]:
decision('No imputar donde no hay huecos. Preservar las banderas de llenado y contrastar capacidad/volumen antes de corregir.', 'Máscaras separadas para falta de texto y anomalía numérica; pérdidas robustas candidatas.', 'La integridad completa del periodo se verifica mediante manifiesto, sin explorar el objetivo de prueba.', 'No es posible inferir MCAR/MAR/MNAR con una matriz numérica sin faltantes. La falta de texto es selección de archivo y fuente; no se presume MCAR.')

### Implicaciones del análisis

- **Preprocesamiento:** No imputar donde no hay huecos. Preservar las banderas de llenado y contrastar capacidad/volumen antes de corregir.
- **Arquitectura:** Máscaras separadas para falta de texto y anomalía numérica; pérdidas robustas candidatas.
- **Validación:** La integridad completa del periodo se verifica mediante manifiesto, sin explorar el objetivo de prueba.
- **Riesgos:** No es posible inferir MCAR/MAR/MNAR con una matriz numérica sin faltantes. La falta de texto es selección de archivo y fuente; no se presume MCAR.